# 04 — Medallion Architecture

**Goal:** Build a reproducible **Bronze → Silver → Gold** batch pipeline using Databricks and Delta Lake.

**Input:** Raw Kaggle CSVs from `/Volumes/workspace/default/retail_raw/`.

**Output:** 6 tables in `workspace.bronze`, 5 in `workspace.silver`, and 3 in `workspace.gold`.

The notebook recreates Silver from Bronze rather than copying the earlier `*_cleaned` tables. It does **not** modify the existing `workspace.default` tables. Reruns overwrite only the named output tables in the three schemas.


## Setup
Create the three schemas and define helpers for qualified table names and Delta writes. Creating schemas requires suitable Unity Catalog permissions.

In [0]:
# Setup: create the three schemas in the workspace catalog
from pyspark.sql import functions as F
from pyspark.sql.window import Window

RAW_VOLUME = "/Volumes/workspace/default/retail_raw"
CATALOG = "workspace"

for layer in ("bronze", "silver", "gold"):
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{layer}")

def table(layer, name):
    return f"{CATALOG}.{layer}.{name}"

def save_delta(df, layer, name):
    target = table(layer, name)
    (
        df.write.format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(target)
    )
    print("Saved:", target)


## 1. Bronze — raw data
Import six source CSVs as Delta tables, retaining source values as strings. No business transformations are applied at this stage.

In [0]:
# Bronze: preserve raw CSV values as strings
raw_files = {
    "sales": "train.csv",
    "stores": "stores.csv",
    "transactions": "transactions.csv",
    "oil": "oil.csv",
    "holidays_events": "holidays_events.csv",
    "sales_test": "test.csv"
}

for name, filename in raw_files.items():
    raw_df = (
        spark.read.option("header", True)
        .option("inferSchema", False)
        .csv(f"{RAW_VOLUME}/{filename}")
    )
    save_delta(raw_df, "bronze", name)


## 2. Silver — standardized data
Read Bronze tables, cast data types, add calendar features, forward-fill observed oil prices, and label effective holidays. The first oil price remains NULL because there is no earlier observation.

In [0]:
# Silver: type casting and basic preparation
sales = (
    spark.table(table("bronze", "sales"))
    .withColumn("id", F.col("id").cast("int"))
    .withColumn("date", F.to_date("date"))
    .withColumn("store_nbr", F.col("store_nbr").cast("int"))
    .withColumn("sales", F.col("sales").cast("double"))
    .withColumn("onpromotion", F.col("onpromotion").cast("int"))
    .withColumn("year", F.year("date"))
    .withColumn("month", F.month("date"))
    .withColumn("day_of_week", F.dayofweek("date"))
)

stores = (
    spark.table(table("bronze", "stores"))
    .withColumn("store_nbr", F.col("store_nbr").cast("int"))
    .withColumn("cluster", F.col("cluster").cast("int"))
)

transactions = (
    spark.table(table("bronze", "transactions"))
    .withColumn("date", F.to_date("date"))
    .withColumn("store_nbr", F.col("store_nbr").cast("int"))
    .withColumn("transactions", F.col("transactions").cast("int"))
)

oil_window = (
    Window.orderBy("date")
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)
oil = (
    spark.table(table("bronze", "oil"))
    .withColumn("date", F.to_date("date"))
    .withColumn("dcoilwtico", F.col("dcoilwtico").cast("double"))
    .withColumn("dcoilwtico", F.last("dcoilwtico", ignorenulls=True).over(oil_window))
)

holidays_events = (
    spark.table(table("bronze", "holidays_events"))
    .withColumn("date", F.to_date("date"))
    .withColumn("transferred", F.col("transferred").cast("boolean"))
    .withColumn(
        "is_effective_holiday",
        ((F.col("type") == "Holiday") & (F.col("transferred") == False)) |
        F.col("type").isin("Additional", "Bridge", "Transfer")
    )
)


## 3. Silver — data quality and persistence
Check required fields, business-key uniqueness, nonnegative sales, and store references before saving the five Silver Delta tables.

In [0]:
# Silver: validate data before saving
def validate(df, name, required, key):
    missing = df.select(
        *[F.sum(F.col(c).isNull().cast("int")).alias(c) for c in required]
    ).first().asDict()
    if any(missing.values()):
        raise ValueError(f"{name} has NULL required fields: {missing}")
    duplicates = df.groupBy(*key).count().filter("count > 1").limit(1).count()
    if duplicates:
        raise ValueError(f"{name} has duplicate business keys: {key}")
    print("Validated:", name)

validate(sales, "sales", ["date", "store_nbr", "family", "sales", "onpromotion"], ["date", "store_nbr", "family"])
validate(stores, "stores", ["store_nbr", "city", "state"], ["store_nbr"])
validate(transactions, "transactions", ["date", "store_nbr", "transactions"], ["date", "store_nbr"])
validate(oil, "oil", ["date"], ["date"])

if sales.filter((F.col("sales") < 0) | (F.col("onpromotion") < 0)).limit(1).count():
    raise ValueError("Negative sales or promotion counts")
if sales.select("store_nbr").distinct().join(stores.select("store_nbr"), "store_nbr", "left_anti").limit(1).count():
    raise ValueError("Sales contains unknown store IDs")

for name, df in {
    "sales": sales,
    "stores": stores,
    "transactions": transactions,
    "oil": oil,
    "holidays_events": holidays_events
}.items():
    save_delta(df, "silver", name)


## 4. Gold — store-level holiday flags
Apply national, regional, and local matching rules, then deduplicate to one row per `(date, store_nbr)`. This table contains **matched holiday dates only**, not every store-day.

In [0]:
# Gold: locale-aware holiday flags
h = spark.table(table("silver", "holidays_events")).filter(F.col("is_effective_holiday"))
st = spark.table(table("silver", "stores"))

holiday_matches = h.alias("h").join(
    st.alias("st"),
    (F.col("h.locale") == "National") |
    ((F.col("h.locale") == "Regional") & (F.col("h.locale_name") == F.col("st.state"))) |
    ((F.col("h.locale") == "Local") & (F.col("h.locale_name") == F.col("st.city"))),
    "inner"
)

holiday_flags = (
    holiday_matches
    .select(F.col("h.date").alias("date"), F.col("st.store_nbr"))
    .distinct()
    .withColumn("is_holiday", F.lit(True))
)
save_delta(holiday_flags, "gold", "store_holiday_flags")


## 5. Gold — daily beverage sales
Create a reporting table with one row per `(date, store_nbr)`. Keep missing transaction counts as NULL rather than replacing them with zero.

In [0]:
# Gold: daily beverage sales with store and holiday context
beverages = (
    spark.table(table("silver", "sales"))
    .filter(F.col("family") == "BEVERAGES")
    .select("date", "store_nbr", F.col("sales").alias("beverage_sales"), "onpromotion")
)

daily_beverage_sales = (
    beverages
    .join(st.select("store_nbr", "city", "state", "type"), "store_nbr", "left")
    .join(holiday_flags, ["date", "store_nbr"], "left")
    .join(
        spark.table(table("silver", "transactions")).select("date", "store_nbr", "transactions"),
        ["date", "store_nbr"], "left"
    )
    .fillna({"is_holiday": False})
)
save_delta(daily_beverage_sales, "gold", "daily_beverage_sales")


## 6. Gold — monthly store-family sales
Create a reusable reporting table with one row per `(year, month, store_nbr, family)`. The measures are **sales quantities, not revenue**.

In [0]:
# Gold: monthly sales by store and product family
monthly_store_family_sales = (
    spark.table(table("silver", "sales"))
    .groupBy("year", "month", "store_nbr", "family")
    .agg(
        F.round(F.sum("sales"), 2).alias("total_sales_qty"),
        F.countDistinct("date").alias("days_with_data"),
        F.round(F.sum("sales") / F.countDistinct("date"), 2).alias("avg_daily_sales_qty"),
        F.sum(F.when(F.col("onpromotion") > 0, 1).otherwise(0)).alias("promotion_days")
    )
    .join(st.select("store_nbr", "city", "state"), "store_nbr", "left")
)
save_delta(monthly_store_family_sales, "gold", "monthly_store_family_sales")


## 7. Verification
Verify that the joins have not multiplied beverage rows. Display final table counts and a sample of the daily Gold table.

In [0]:
# Verify counts and uniqueness after Gold joins
before = beverages.count()
daily_gold = spark.table(table("gold", "daily_beverage_sales"))
after = daily_gold.count()
assert before == after, f"Unexpected row duplication: {before} vs {after}"

duplicates = (
    daily_gold.groupBy("date", "store_nbr")
    .count().filter("count > 1").limit(1).count()
)
assert duplicates == 0, "Duplicate (date, store_nbr) rows in daily beverage Gold table"

for name in ("store_holiday_flags", "daily_beverage_sales", "monthly_store_family_sales"):
    print(table("gold", name), "rows:", spark.table(table("gold", name)).count())

display(daily_gold.orderBy("date", "store_nbr").limit(10))


## Outputs

| Layer | Schema | Tables |
|---|---|---|
| Bronze | `workspace.bronze` | `sales`, `stores`, `transactions`, `oil`, `holidays_events`, `sales_test` |
| Silver | `workspace.silver` | `sales`, `stores`, `transactions`, `oil`, `holidays_events` |
| Gold | `workspace.gold` | `store_holiday_flags`, `daily_beverage_sales`, `monthly_store_family_sales` |

**Notes:** This is a full-refresh batch pipeline. Only the listed tables in the three schemas are overwritten. The holiday-flag table contains matching holidays only. The Gold sales measures are quantities, not monetary revenue. Model-ready features and a star schema are reserved for later notebooks.
